In [31]:
import pandas as pd
import numpy as np
import ast
import psycopg2
import json
from psycopg2.extras import execute_values
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm


DB_CONFIG = {
    "host": "localhost", 
    "port": "5432",
    "database": "movies_db",
    "user": "user",
    "password": "password"
}



model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

# Загрузка данных

In [33]:
def extract_genres(genre_str):
    try:
        genres_list = ast.literal_eval(genre_str)
        return ", ".join([g['name'] for g in genres_list])
    except:
        return ""

print("Загрузка и обработка CSV...")
df = pd.read_csv('./data/lesson36_data/movies_metadata.csv', low_memory=False, on_bad_lines='skip')

# Оставляем только нужные колонки и чистим пропуски
df = df[['title', 'overview', 'genres']].dropna(subset=['title', 'overview'])
df = df.drop_duplicates(subset=['overview']).reset_index(drop=True)

# Парсим жанры и склеиваем текст
df['clean_genres'] = df['genres'].apply(extract_genres)
df['combined_text'] = "Title: " + df['title'] + ". Genres: " + df['clean_genres'] + ". Overview: " + df['overview']

# Берем 10 000 фильмов для комфортной работы на CPU
print(f"Подготовлено фильмов: {len(df)}")

Загрузка и обработка CSV...
Подготовлено фильмов: 44303


# Загрука модели

In [25]:
model_name = 'all-MiniLM-L6-v2'
print(f"Загрузка модели {model_name} ")
model = SentenceTransformer(model_name, device='cpu')


Загрузка модели all-MiniLM-L6-v2 


# Генерация емебендингов

In [34]:
embeddings = model.encode(
    df['combined_text'].tolist(), 
    batch_size=64, 
    show_progress_bar=True, 
    convert_to_numpy=True
)
print(f"Матрица векторов готова: {embeddings.shape}")

Batches:   0%|          | 0/693 [00:00<?, ?it/s]

Матрица векторов готова: (44303, 384)


# Загрузка данных в PostgreSQL

In [35]:
def upload_data():
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    
    # Подготовка данных для SQL (список кортежей)
    records = []
    for i in range(len(df)):
        records.append((
            df.iloc[i]['title'],
            df.iloc[i]['clean_genres'],
            df.iloc[i]['overview'],
            df.iloc[i]['combined_text'],
            embeddings[i].tolist() # Превращаем numpy в список для БД
        ))
    
    insert_query = """
        INSERT INTO movies (title, genres, overview, combined_text, embedding)
        VALUES %s
    """
    
    # Заливаем в базу батчами по 500 штук
    batch_size = 500
    for i in tqdm(range(0, len(records), batch_size), desc="Запись в БД"):
        batch = records[i:i+batch_size]
        execute_values(cur, insert_query, batch)
        conn.commit()
    
    cur.close()
    conn.close()
    print("Данные успешно перенесены в PostgreSQL!")



In [40]:
upload_data()

Запись в БД:   0%|          | 0/89 [00:00<?, ?it/s]

Данные успешно перенесены в PostgreSQL!


# Функция рекомендаций

In [ ]:
def get_recommendations_and_log(source_title, top_n=5):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    
    # 1. Сначала достаем информацию о выбранном фильме из базы
    cur.execute("SELECT combined_text FROM movies WHERE title = %s LIMIT 1;", (source_title,))
    result = cur.fetchone()
    if not result:
        return "Фильм не найден в базе."
    
    source_text = result[0]
    
    # 2. Генерируем вектор для этого фильма
    query_vector = model.encode([source_text])[0].tolist()
    
    # 3. Векторный поиск в БД (оператор <=> считает косинусное расстояние)
    search_query = """
        SELECT title, genres, 1 - (embedding <=> %s::vector) AS similarity
        FROM movies
        WHERE title != %s
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
    """
    cur.execute(search_query, (query_vector, source_title, query_vector, top_n))
    recs = cur.fetchall()
    
    recommendations_data = []
    print(f"\nРекомендации для фильма: {source_title}\n" + "-"*40)
    
    for row in recs:
        title, genres, sim = row
        score_percent = round(max(0, sim) * 100, 1)
        print(f"{title} ({score_percent}%) | Жанры: {genres}")
        recommendations_data.append({"title": title, "match": score_percent})
    
    log_query = """
        INSERT INTO recommendation_logs (source_title, source_text, recommendations)
        VALUES (%s, %s, %s)
    """
    cur.execute(log_query, (source_title, source_text, json.dumps(recommendations_data)))
    conn.commit()
    
    cur.close()
    conn.close()
    print("\n[Результат сохранен в БД в таблицу recommendation_logs]")

In [66]:
random_movie = df['title'].sample(5).values[0]
get_recommendations_and_log(random_movie)


Рекомендации для фильма: Desert Winds
----------------------------------------
The Wind (57.2%) | Жанры: Drama, Romance, Western
Up in the Wind (53.0%) | Жанры: 
Dust in the Wind (52.3%) | Жанры: Drama, Romance
Black Wind (51.6%) | Жанры: Adventure, Drama
Written on the Wind (51.1%) | Жанры: Drama, Romance

[Результат сохранен в БД в таблицу recommendation_logs]


In [67]:
import pandas as pd
import numpy as np
import psycopg2
import sentence_transformers
import tqdm
import sklearn

print(f"pandas=={pd.__version__}")
print(f"numpy=={np.__version__}")
print(f"psycopg2=={psycopg2.__version__}")
print(f"sentence-transformers=={sentence_transformers.__version__}")
print(f"tqdm=={tqdm.__version__}")
print(f"scikit-learn=={sklearn.__version__}")

pandas==2.3.3
numpy==1.26.4
psycopg2==2.9.11 (dt dec pq3 ext lo64)
sentence-transformers==5.2.2
tqdm==4.67.1
scikit-learn==1.8.0


In [ ]:


pd.set_option('display.max_colwidth', None)

conn = psycopg2.connect(
    host="localhost",
    port="5432",
    dbname="movies_db",
    user="user",
    password="password"
)

query = """
    SELECT 
        source_title AS "Искомый фильм", 
        created_at::timestamp(0) AS "Время",
        recommendations AS "Подобранные рекомендации"
    FROM recommendation_logs 
    ORDER BY created_at DESC 
    LIMIT 10;
"""

logs_df = pd.read_sql(query, conn)
conn.close()

def format_recommendations(recs):
    if isinstance(recs, list):
        formatted_items = []
        for item in recs:
            title = item.get('title', 'Неизвестно')
            match = item.get('match', 0)
            formatted_items.append(f"{title} ({match}%)")
        return " | ".join(formatted_items)
    return recs

logs_df['Подобранные рекомендации'] = logs_df['Подобранные рекомендации'].apply(format_recommendations)

display(logs_df)

C:\Users\Kirill\AppData\Local\Temp\ipykernel_26856\1700559047.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  logs_df = pd.read_sql(query, conn)


,Искомый фильм,Время,Подобранные рекомендации
0,Desert Winds,2026-03-11 18:18:08,The Wind (57.2%) | Up in the Wind (53.0%) | Dust in the Wind (52.3%) | Black Wind (51.6%) | Written on the Wind (51.1%)
1,"My Son, My Son, What Have Ye Done",2026-03-11 18:17:48,"All My Sons (62.0%) | The Good Son (61.8%) | The Son (58.4%) | Edward, My Son (58.0%) | Youth of the Son (57.0%)"
2,Double Play: James Benning and Richard Linklater,2026-03-11 18:17:36,21 Years: Richard Linklater (63.6%) | Slacker 2011 (61.9%) | Side by Side (55.7%) | 1 P.M. (52.6%) | Not Quite Hollywood (52.0%)
3,Make Way For A Lady,2026-03-11 18:16:53,The Making of a Lady (59.6%) | Should Ladies Behave (57.0%) | The Matchmaker (56.0%) | The Lady and the Highwayman (53.0%) | Just the Way You Are (52.6%)
4,Alice,2026-03-11 18:10:02,More of Me (65.4%) | Alice Upside Down (61.2%) | Alice Adams (58.1%) | Alice in Wonderland (57.6%) | Alice and Martin (57.5%)
5,Song of the Saddle,2026-03-11 18:09:57,Back in the Saddle (64.0%) | Man in the Saddle (62.4%) | Boots and Saddles (61.0%) | Blazing Saddles (59.5%) | Tall in the Saddle (59.4%)
6,Nasha Russia: Yaytsa sudby,2026-03-11 17:43:29,Satisfaktsiya (84.5%) | Pro Lyuboff (82.0%) | Moscow (81.2%) | Dura (79.7%) | Bablo (78.6%)
7,Another Woman,2026-03-11 17:43:25,Possessed (55.8%) | Film About a Woman Who… (55.8%) | Felt (55.7%) | Between Strangers (55.6%) | The Other Woman (55.5%)
8,Directed by Sidney Lumet: How the Devil Was Made,2026-03-11 17:43:09,"Before the Devil Knows You're Dead (63.7%) | The Old Devil (63.7%) | The Devil, Probably (59.8%) | Beat the Devil (54.1%) | Proof of the Devil (53.4%)"
